In [1]:
import pandas as pd
import os

In [2]:
customers = pd.read_csv("/Users/janhavisingh/personal_projects/retail-personalisation-system/data/raw/customers.csv")
articles = pd.read_csv("/Users/janhavisingh/personal_projects/retail-personalisation-system/data/raw/articles.csv")
sample_submission = pd.read_csv("/Users/janhavisingh/personal_projects/retail-personalisation-system/data/raw/sample_submission.csv")
transactions_train = pd.read_csv("/Users/janhavisingh/personal_projects/retail-personalisation-system/data/raw/transactions_train.csv")

### 1. Split the transactions by time

The objective is to evaluate how effectively a recommendation system can predict articles that a customer will purchase in a future 7-day period.

For each customer, the recommender will generate up to 12 article recommendations using only information available before the evaluation period.

Recommendation performance will initially be measured using Recall@12 and MAP@12. Simple popularity-based approaches will be established as baselines before evaluating personalised recommendation models.

In [3]:
transactions_train["t_dat"] = pd.to_datetime(transactions_train["t_dat"])
max_date = transactions_train["t_dat"].max()

eval_start = max_date - pd.Timedelta(days=6)
train = transactions_train[
    transactions_train["t_dat"] < eval_start
].copy()

evaluation = transactions_train[
    transactions_train["t_dat"] >= eval_start
].copy()

print("Training:", train["t_dat"].min(), "to", train["t_dat"].max())
print("Evaluation:", evaluation["t_dat"].min(), "to", evaluation["t_dat"].max())

Training: 2018-09-20 00:00:00 to 2020-09-15 00:00:00
Evaluation: 2020-09-16 00:00:00 to 2020-09-22 00:00:00


### 2. Ground-truth construction

In [4]:
ground_truth = (
    evaluation
    .groupby("customer_id")["article_id"]
    .apply(set)
    .reset_index(name="actual_articles")
)

ground_truth.head()

,customer_id,actual_articles
0,00039306476aaf41a07fed942884f16b30abfa83a2a8be...,{624486001}
1,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf...,{827487003}
2,000493dd9fc463df1acc2081450c9e75ef8e87d5dd17ed...,"{757926001, 640021019, 788575004}"
3,000525e3fe01600d717da8423643a8303390a055c578ed...,{874110016}
4,00077dbd5c4a4991e092e63893ccf29294a9d5c46e8501...,"{158340001, 935892001, 799365027, 918171001, 9..."


### 4. Global popularity baseline

In [5]:
global_popular = train["article_id"].value_counts().head(12)
global_recommendations = global_popular.index.tolist()
global_recommendations

[706016001,
 706016002,
 372860001,
 610776002,
 759871002,
 464297007,
 372860002,
 610776001,
 399223001,
 720125001,
 706016003,
 156231001]

### 5. Recent popularity baseline

In [6]:
recent_start = eval_start - pd.Timedelta(days=7)

recent_trans = train[
    train["t_dat"] >= recent_start
].copy()

recent_popular = recent_trans["article_id"].value_counts().head(12)
recent_recommendations = recent_popular.index.tolist()
recent_recommendations

[909370001,
 865799006,
 918522001,
 924243001,
 448509014,
 751471001,
 809238001,
 918292001,
 762846027,
 809238005,
 673677002,
 923758001]

### 6. Recall@12 and MAP@12

#### Recall@12
Actual purchases:       {A, B, C, D}

Recommended top 12:     [X, B, Y, A, Z, ...]

Hits = {A, B}

Recall@12 = 2 / 4 = 0.50 

Of everything this customer actually bought, what proportion did our 12 recommendations manage to retrieve?

In [7]:
def recall_at_k(actual, recommended, k=12):
    recommended_k = recommended[:k]
    hits = len(set(actual) & set(recommended_k))
    return hits / len(actual)

ground_truth["global_recall"] = ground_truth["actual_articles"].apply(
    lambda x: recall_at_k(x, global_recommendations)
)
global_recall = ground_truth["global_recall"].mean()

ground_truth["recent_recall"] = ground_truth["actual_articles"].apply(
    lambda x: recall_at_k(x, recent_recommendations)
)
recent_recall = ground_truth["recent_recall"].mean()

print(global_recall)
print(recent_recall)

0.007748996773688122
0.02550428716366145


##### MAP@12

MAP is slightly more sophisticated because it cares about where the correct recommendations appear.
Suppose the customer actually buys {A, B}.

model 1 [A, B, X, Y, Z, ...]
         ↑  ↑

model 2 [X, Y, Z, A, B, ...]
                  ↑  ↑

Both models found the same two relevant products, so their Recall@12 could be identical.

But Model 1 is better because the relevant products were ranked #1 and #2 rather than #4 and #5.

That's what Average Precision captures.

In [8]:
def average_precision_at_k(actual, recommended, k=12):
    actual = set(actual)
    recommended_k = recommended[:k]

    score = 0.0
    hits = 0

    for rank, article in enumerate(recommended_k, start=1):
        if article in actual:
            hits += 1
            precision_at_rank = hits / rank
            score += precision_at_rank

    return score / min(len(actual), k)

ground_truth["global_ap@12"] = ground_truth["actual_articles"].apply(
    lambda actual: average_precision_at_k(
        actual,
        global_recommendations,
        k=12
    )
)
global_map = ground_truth["global_ap@12"].mean()

ground_truth["recent_ap@12"] = ground_truth["actual_articles"].apply(
    lambda actual: average_precision_at_k(
        actual,
        recent_recommendations,
        k=12
    )
)
recent_map_7 = ground_truth["recent_ap@12"].mean()

print(global_map)
print(recent_map_7)

0.0028985916189694194
0.008747681312782597


### 7. Baseline comparison

In [9]:
lookback_windows = [7, 14, 28]

baseline_results = []


baseline_results.append({
    "Baseline": "Global popularity",
    "Recall@12": global_recall,
    "MAP@12": global_map
})


# Recent popularity windows
for days in lookback_windows:

    recent_start = eval_start - pd.Timedelta(days=days)

    recent_transactions = train[
        train["t_dat"] >= recent_start
    ]

    recent_recommendations = (
        recent_transactions["article_id"]
        .value_counts()
        .head(12)
        .index
        .tolist()
    )

    recall = ground_truth["actual_articles"].apply(
        lambda x: recall_at_k(x, recent_recommendations)
    ).mean()

    map_score = ground_truth["actual_articles"].apply(
        lambda x: average_precision_at_k(
            x,
            recent_recommendations
        )
    ).mean()

    baseline_results.append({
        "Baseline": f"Recent popularity - {days}d",
        "Recall@12": recall,
        "MAP@12": map_score
    })


baseline_comparison = pd.DataFrame(baseline_results)

baseline_comparison

,Baseline,Recall@12,MAP@12
0,Global popularity,0.007749,0.002899
1,Recent popularity - 7d,0.025504,0.008748
2,Recent popularity - 14d,0.024996,0.008005
3,Recent popularity - 28d,0.019322,0.005482


### 8. Key findings

Recent popularity is much better than global popularity on Recall@12.

So the recent-popularity baseline is retrieving about 2.55% of customers’ actually purchased items, compared with only about 0.78% for global popularity.
That is roughly a 3.3× improvement in recall.

The Recall result itself already tells us something interesting about the domain: product demand appears quite time-sensitive. Articles that were popular over the entire historical period are considerably less useful for predicting next week's purchases than articles popular immediately before the prediction date. That makes sense for fashion, where availability, seasonality and trends change.


Recent popularity substantially outperformed global popularity.

- Global popularity achieved Recall@12 of 0.0077 and MAP@12 of 0.0029.
- Recent popularity achieved Recall@12 of 0.0255 and MAP@12 of 0.0087.

This suggests that short-term purchasing trends are considerably more informative for next-week recommendations than popularity measured across the full historical period.

The absolute scores remain low, which is expected because both baselines provide the same recommendations to every customer and do not use individual customer preferences.

- Performance decreased as the popularity lookback window increased from 7 to 14 and 28 days, suggesting that article relevance changes relatively quickly and recent purchasing behaviour is particularly informative.
- Despite outperforming global popularity, recent popularity remains a non-personalised approach: every customer receives the same recommendations.
- The 7-day recent-popularity model will therefore be used as the primary baseline against which personalised recommendation approaches are evaluated.